In [1]:
import numpy as np
import photolab as lab
import torch
from PIL import Image
from transformers import AutoModel, AutoProcessor

torch.set_grad_enabled(False)  # inference only: no learning, less memory
model = AutoModel.from_pretrained(lab.MODEL_DIR).eval()
processor = AutoProcessor.from_pretrained(lab.MODEL_DIR)
scale = model.logit_scale.exp().item()
bias = model.logit_bias.item()
print(type(model).__name__, "| logit scale", round(scale, 2), "| logit bias", round(bias, 2))

/Users/rushilshandil/.cache/uv/archive-v0/82JjMR0gVF3nCFwD/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.
Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 408/408 [00:00<00:00, 12285.88it/s]


SiglipModel | logit scale 117.33 | logit bias -12.93


In [2]:
def as_tensor(out):
    # transformers 5 wraps the features; older versions return the tensor itself
    return getattr(out, "pooler_output", out)


def embed_images(paths, batch=8):
    rows = []
    for start in range(0, len(paths), batch):
        images = [Image.open(p).convert("RGB") for p in paths[start : start + batch]]
        pixels = processor(images=images, return_tensors="pt")["pixel_values"]
        vectors = as_tensor(model.get_image_features(pixel_values=pixels))
        rows.append(torch.nn.functional.normalize(vectors, dim=-1))
    return torch.cat(rows).numpy()


def embed_texts(texts):
    tokens = processor(text=texts, padding="max_length", max_length=64, return_tensors="pt")
    vectors = as_tensor(model.get_text_features(input_ids=tokens["input_ids"]))
    return torch.nn.functional.normalize(vectors, dim=-1).numpy()


def match_probability(cosine):
    return 1.0 / (1.0 + np.exp(-(cosine * scale + bias)))

In [3]:
lab.CACHE.mkdir(exist_ok=True)
index = {}
for client in sorted({c.client for c in lab.load_cases()}):
    paths = lab.photos_of(client)
    vectors = embed_images(paths)
    np.save(lab.CACHE / f"{client}.npy", vectors)
    index[client] = ([p.name for p in paths], vectors)
    print(
        client, vectors.shape, "length of the first:", round(float(np.linalg.norm(vectors[0])), 4)
    )

kowalski-plumbing (8, 768) length of the first: 1.0
lone-star-roofing (8, 768) length of the first: 1.0
sparkle-squad (8, 768) length of the first: 1.0


In [4]:
def siglip_searcher(template="{}"):
    def search(case):
        names, photos = index[case.client]
        query = embed_texts([template.format(case.query)])[0]
        cosines = photos @ query  # one dot product per photo
        probs = match_probability(cosines)
        return names, cosines, float(probs.max())

    return search


cases = lab.load_cases()
search = siglip_searcher()
print(lab.evaluate(cases, search))

{'recall@1': 0.9705882352941176, 'recall@3': 1.0, 'mrr': 0.9852941176470589, 'cases': 48.0}


In [5]:
import pandas as pd

templates = {
    "as typed": "{}",
    "a photo of ...": "a photo of {}.",
    "a photo of the ...": "a photo of the {}.",
}
pd.DataFrame(
    {name: lab.evaluate(cases, siglip_searcher(t)) for name, t in templates.items()}
).T.round(3)

,recall@1,recall@3,mrr,cases
as typed,0.971,1.0,0.985,48.0
a photo of ...,1.000,1.0,1.000,48.0
a photo of the ...,0.941,1.0,0.971,48.0


In [6]:
honest = lab.leave_one_client_out(cases, siglip_searcher("a photo of {}."))
honest

{'right_action': 0.8333333333333334,
 'thresholds': [0.0015075046103447676,
  0.0021427262108772993,
  0.000516675238031894]}

In [ ]:
import matplotlib.pyplot as plt


def show(case, search, top=3):
    names, scores, confidence = search(case)
    order = np.argsort(-scores)[:top]
    fig, axes = plt.subplots(1, top, figsize=(3 * top, 3))
    fig.suptitle(
        f"{case.query!r}  (confidence {confidence:.2f}, want {list(case.want) or 'nothing'})"
    )
    for ax, i in zip(axes, order, strict=False):  # fewer photos than `top`: fewer panels
        ax.imshow(Image.open(lab.PHOTOS / case.client / names[i]))
        ax.set_title(
            f"{names[i]}  {scores[i]:.3f}", color="green" if names[i] in case.want else "black"
        )
        ax.axis("off")
    plt.show()


search = siglip_searcher("a photo of {}.")
for case in cases[:5]:
    show(case, search)

In [9]:
search = siglip_searcher("a photo of {}.")
clients = sorted({c.client for c in cases})
for client, line in zip(clients, honest["thresholds"], strict=True):
    for case in (c for c in cases if c.client == client):
        names, scores, confidence = search(case)
        swaps = confidence >= line
        top = names[int(np.argmax(scores))]
        right = (swaps and top in case.want) if case.want else not swaps
        if not right:
            what = f"swapped in {top}" if swaps else "asked"
            print(
                f"{client} (line {line:.4f}) {case.query!r}: confidence {confidence:.4f}, "
                f"{what}; want {list(case.want) or 'nothing'}"
            )

lone-star-roofing (line 0.0021) 'man in check shirt': confidence 0.0005, asked; want ['lone-star-roofing-04.jpg']
lone-star-roofing (line 0.0021) 'man wearing helmet': confidence 0.0015, asked; want ['lone-star-roofing-04.jpg']
lone-star-roofing (line 0.0021) 'hail damage on shingles': confidence 0.0066, swapped in lone-star-roofing-03.jpg; want nothing
lone-star-roofing (line 0.0021) 'a drone shot of a whole roof from above': confidence 0.0733, swapped in lone-star-roofing-07.jpg; want nothing
lone-star-roofing (line 0.0021) 'a metal roof on a barn': confidence 0.0079, swapped in lone-star-roofing-03.jpg; want nothing
sparkle-squad (line 0.0005) 'a carpet being steam cleaned': confidence 0.0007, swapped in sparkle-squad-01.jpg; want nothing
sparkle-squad (line 0.0005) 'an empty office after cleaning': confidence 0.0014, swapped in sparkle-squad-01.jpg; want nothing
sparkle-squad (line 0.0005) 'a dog asleep on a clean sofa': confidence 0.0028, swapped in sparkle-squad-06.jpg; want noth

In [10]:
memo = {}


def cached(case):
    # each case is searched once; choosing a line then costs nothing
    key = (case.client, case.query)
    if key not in memo:
        memo[key] = search(case)
    return memo[key]


def outcome(case, line):
    names, scores, confidence = cached(case)
    if confidence < line:
        return "needless ask" if case.want else "right"
    return "right" if names[int(np.argmax(scores))] in case.want else "wrong swap"


COST = {"right": 0.0, "needless ask": 1.0, "wrong swap": 3.0}


def cheapest_line(train):
    lines = [0.0, *sorted({cached(c)[2] for c in train}), 1.0]
    return min(lines, key=lambda t: sum(COST[outcome(c, t)] for c in train))


policies = {
    "most right actions": lambda train: lab.best_threshold(train, cached),
    "a wrong swap costs 3 asks": cheapest_line,
}
rows = []
for policy, choose in policies.items():
    for client in clients:
        line = choose([c for c in cases if c.client != client])
        for case in (c for c in cases if c.client == client):
            rows.append({"policy": policy, "client": client, "outcome": outcome(case, line)})
table = pd.DataFrame(rows)
pd.crosstab(table.policy, table.outcome)

outcome,needless ask,right,wrong swap
policy,,,
a wrong swap costs 3 asks,9,36,3
most right actions,2,40,6


In [11]:
print("line for the product:", cheapest_line(cases))

line for the product: 0.008523262105882168
